In [ ]:
from evolve.evolvetest import digivolve_multi
import pandas as pd
from Bio import SeqIO
import os
import itertools
def get_stuff(prot_name):
    record = SeqIO.read(('../EvolveTest/wt_fasta/'+prot_name+'_WT.fasta'),"fasta")
    wt_aa_seq = record.seq
    embeddings_dir = '../EvolveTest/' + prot_name
    embeddings_models = [
        'esmc_300m_multi',
        #'esmc_600m_multi'
    ]
    embeddings_paths = list(map(lambda x: ('embeddings/'+ prot_name + '_' + x + '.csv'),embeddings_models))

    
    return wt_aa_seq, embeddings_dir, embeddings_paths

layers = [500,100,50]
rounds_evo = 10
num_var = 16
results = pd.DataFrame()
# Define all possible datasets
all_datasets = {
    'CLYGR': ('CLYGR', r'..\EvolveTest\DMS_data\DMS_ProteinGym_substitutions\D7PM05_CLYGR_Somermeyer_2022.xlsx', 12500),
    #'AAV2S': ('AAV2S', r'..\EvolveTest\DMS_data\DMS_ProteinGym_substitutions\CAPSD_AAV2S_Sinai_2021.xlsx', -1.2),
    #'HIS7': ('HIS7', r'..\EvolveTest\DMS_data\DMS_ProteinGym_substitutions\HIS7_YEAST_Pokusaeva_2019.xlsx', 0.3),
    #'GRB2': ('GRB2', r'..\EvolveTest\DMS_data\DMS_ProteinGym_substitutions\GRB2_HUMAN_Faure_2021.xlsx', -0.7), #max 2
    #'SPG1': ('SPG1', r'..\EvolveTest\DMS_data\DMS_ProteinGym_substitutions\SPG1_STRSG_Olson_2014(2).xlsx', -4), #max 2
}

for protein_id, data in all_datasets.items():
    protein_code = data[0]
    exp_activity_file_path = data[1]
    wt_activity = data[2]
    
    # Get necessary data for processing (do this once per protein)
    wt_aa_seq, embeddings_dir, embeddings_paths = get_stuff(protein_code)
    logits_path = os.path.join(embeddings_dir, 'embeddings', f'logits_{protein_code}_wt_melted.csv')
    
    # Then iterate through rounds
    for rounds_to_advance in range(1, 3):
        # Then iterate through scope-depth pairs
        for pair in itertools.combinations(range(1, 2), 2):
            scope = pair[0]
            depth = pair[1]
            
            # Determine which variable to use
            if rounds_to_advance:
                if scope != 1:
                    continue
                variable = 'rounds'
            else:
                variable = scope
            
            # Create dataset name
            dataset_name = f'{rounds_to_advance}_{protein_id}_{depth}_{variable}'
            print(dataset_name)
            # Run the digivolve function
            result = digivolve_multi(
                wt_aa_seq, protein_code,
                wt_activity, dataset_name, exp_activity_file_path,
                embeddings_dir, embeddings_paths,
                num_var, layers,
                logits_path, scope, depth, rounds_to_advance
            )
            
            # Append results
            results = pd.concat([results, result], axis=1)
    print(results)

Empty DataFrame
Columns: []
Index: []


In [ ]:
from evolve.evolvetest import digivolveZS
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb
from Bio import SeqIO
import os
randomforest = RandomForestRegressor(
        n_estimators=100,
        criterion="friedman_mse",
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        min_weight_fraction_leaf=0.0,
        max_features=1.0,
        max_leaf_nodes=None,
        min_impurity_decrease=0.0,
        bootstrap=True,
        oob_score=False,
        n_jobs=None,
        random_state=1,
        verbose=0,
        warm_start=False,
        ccp_alpha=0.0,
        max_samples=None,
    )
estimators = [
    ('mlp', MLPRegressor(random_state=1,max_iter=5000)),
    ('gbr', GradientBoostingRegressor(random_state=1))
]
def get_stuff(prot_name):
    record = SeqIO.read(('../EvolveTest/wt_fasta/'+prot_name+'_WT.fasta'),"fasta")
    wt_aa_seq = record.seq
    embeddings_dir = '../EvolveTest/' + prot_name
    embeddings_models = [
        #'esm1b_t33_650M_UR50S',
        #'esm1v_t33_650M_UR90S_1',
        #'esm1v_t33_650M_UR90S_2',
        #'esm1v_t33_650M_UR90S_3',
        #'esm1v_t33_650M_UR90S_4',
        #'esm1v_t33_650M_UR90S_5',
        #'esm2_t6_8M_UR50D',
        #'esm2_t12_35M_UR50D',
        #'esm2_t30_150M_UR50D',
        #'esm2_t33_650M_UR50D',
        #'esm2_t48_15B_UR50D',
        'esmc_300m_WT',
        #'esmc_600m_WT'
    ]
    embeddings_paths = list(map(lambda x: ('embeddings/'+ prot_name + '_' + x + '.csv'),embeddings_models))

    
    return wt_aa_seq, embeddings_dir, embeddings_paths
zero_shot = [
            #[0,0,16], #[marginal WT ranked, raw prob ranked, random]
            [0,16,0],
            #[12,0,4],
            #[8,8,0],
             ]
model = 'NeuralNet'
models = {
    'ExtraTrees' : ExtraTreesRegressor(criterion="friedman_mse",random_state=1),
    'RandomForest': randomforest,
    'GradientBoosting': GradientBoostingRegressor(random_state=1),
    'SVR': SVR(),
    'NeuralNet': MLPRegressor(random_state=1,max_iter=5000, hidden_layer_sizes=(500,100,50)),
    'XGBoost': xgb.XGBRegressor(random_state=1),
    'LightGBM': lgb.LGBMRegressor(random_state=1),
    'StackingRegressor': StackingRegressor(estimators=estimators,
                                            final_estimator=SVR()),
    'NNStackingRegressor': StackingRegressor(estimators=estimators,
                                             final_estimator=randomforest)
}
model_type = models[model]
wt_activity = 1
rounds_evo =  10
num_var = 16
prot_name = 'brenan'
wt_aa_seq, embeddings_dir, embeddings_paths = get_stuff(prot_name)
logits_path = os.path.join(embeddings_dir,'embeddings',f'logits_{prot_name}_wt_melted.csv')
runs = {
    #f'zika_NN_500_rand':r'..\EvolveTest\DMS_data\zika_dms.xlsx',
    #f'rubisco_NN_full':r'..\EvolveTest\DMS_data\rubisco_dms.xlsx',
    #f'AmiE_Acet_NN_full':r'..\EvolveTest\DMS_data\amie_Acet_dms.xlsx', 
    #f'AmiE_Isobut_NN_full':r'..\EvolveTest\DMS_data\amie_Isobut_dms.xlsx',
    #f'AmiE_Propi_NN_full':r'..\EvolveTest\DMS_data\amie_Propi_dms.xlsx',
    #f'Tem_2500_NN_500':r'..\EvolveTest\DMS_data\bla_amp_2500_dms.xlsx',
    #f'Tem_cef_NN_500':r'..\EvolveTest\DMS_data\bla_cefo_dms.xlsx',
    f'brenan_DOX_500':r'..\EvolveTest\DMS_data\brenan_DOX_dms.xlsx', 
    f'brenan_SCH_500':r'..\EvolveTest\DMS_data\brenan_SCH_dms.xlsx',
    f'brenan_VRT_500':r'..\EvolveTest\DMS_data\brenan_VRT_dms.xlsx',
    }
for name, dms_data in runs.items():
    dataset = name
    print(dataset)
    exp_activity_file_path = dms_data
    #plot_combined_landscape(exp_activity_file_path,embeddings_paths[-1],logits_path,prot_name)
    digivolveZS(wt_aa_seq,prot_name,
                          wt_activity,dataset,
                          exp_activity_file_path,
                          embeddings_dir, embeddings_paths,
                          num_var,rounds_evo,
                          model_type,
                          logits_path, zero_shot[0],new_variable = model)

brenan_VRT_500


Processing embedding paths:   0%|          | 0/1 [00:00<?, ?it/s]

Evolution rounds for embeddings/brenan_esmc_300m_WT.csv:   0%|          | 0/10 [00:00<?, ?it/s]

Evolution Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Evolution Progress:   0%|          | 0/100 [00:00<?, ?it/s]

...

Evolution Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Evolution Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Evolution Progress:   0%|          | 0/100 [00:00<?, ?it/s]

Evolution Progress:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\ese\miniforge3\envs\foldy\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


In [ ]:
import pandas as pd
from evolve.evolvetest import plot_evolution_boxplot
benchmarks = {
        'Above WT' : 1,
        '50th Percentile' : 0.5,
        '90th Percentile' : 0.9,
        '95th Percentile' : 0.95,
        'Number of Var>WT' : 0,
        'Avg var per round(*10)':0,
}
df = results
df.to_excel(os.path.join()'CLYGR Full')
print(df)
plot_evolution_boxplot(df, benchmarks, new_variable='',save_path=None)

          embeddings/CLYGR_esmc_300m_multi.csv  \
0  [1, 1, 1, 2, None, 11, 0.994, 2.787, 0.105]   
1      [1, 1, 3, 3, 3, 5, 0.999, 3.011, 0.163]   
2      [1, 1, 1, 4, 4, 4, 0.996, 2.839, 0.178]   
3        [1, 1, 3, 3, 3, 3, 0.997, 2.922, 0.1]   
4       [1, 1, 2, 4, 4, 5, 0.995, 2.84, 0.134]   
5    [1, 1, 3, 9, None, 6, 0.984, 2.666, 0.23]   
6  [1, 1, 1, 2, None, 11, 0.991, 2.737, 0.137]   
7     [1, 1, 1, 2, 9, 11, 0.996, 2.848, 0.116]   
8   [1, 1, 2, 2, None, 8, 0.994, 2.787, 0.241]   
9   [1, 1, 1, 2, None, 7, 0.983, 2.655, 0.126]   

          embeddings/CLYGR_esmc_300m_multi.csv  \
0  [1, 1, 2, 2, None, 10, 0.988, 2.709, 0.311]   
1      [1, 1, 4, 4, 4, 2, 0.998, 2.956, 0.142]   
2      [1, 1, 1, 2, 5, 5, 0.999, 2.978, 0.206]   
3      [1, 1, 1, 4, 7, 8, 0.996, 2.843, 0.262]   
4  [1, 1, 1, 4, None, 10, 0.995, 2.806, 0.231]   
5        [1, 1, 3, 3, 8, 5, 0.997, 2.91, 0.14]   
6        [1, 1, 2, 2, 4, 2, 0.998, 2.91, 0.12]   
7      [1, 1, 1, 3, 10, 6, 0.998, 2.91, 0.146]   